In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model, optimizers
import numpy as np
import os

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
WEIGHTS_OUTPUT_PATH = os.path.join(BASE_PATH, "SIMCLR_Encoder.weights.h5")

# Config
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.0001
TEMPERATURE = 0.1
PROJECTION_DIM = 64
LATENT_DIM = 128

# --- 1. Data Augmentation (The "Uni-Modal" Trick) ---
def augment_payload(x):
    """
    Simulates network packet loss/corruption by randomly masking 15% of bytes.
    Input: (Batch, 7840, 1) or (Batch, 10, 784)
    """
    # Create a binary mask (1 = keep, 0 = drop)
    # We use Dropout logic to effectively 'mask' inputs
    return tf.nn.dropout(x, rate=0.15)

# --- 2. Data Loading ---
def load_data():
    if not os.path.exists(RAW_PAYLOAD_PATH):
        print(f"Error: {RAW_PAYLOAD_PATH} not found.")
        return None
    X = np.load(RAW_PAYLOAD_PATH).astype('float32')

    # Reshape for CNN (N, 7840, 1) - Flattening packets to 1 stream
    # Note: Ensure this matches the shape used in your main LE-MVCL
    if len(X.shape) == 3: # If (N, 10, 784)
        X = X.reshape(X.shape[0], -1, 1)
    else:
        X = X.reshape(X.shape[0], -1, 1)

    print(f"Loaded Raw Data: {X.shape}")
    return X

# --- 3. CNN Architecture (Must match LE-MVCL exactly) ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)

    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)

    # Projection Head (Only for training)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1, epsilon=1e-10))(z)

    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 4. SimCLR Model Wrapper ---
class SimCLR(Model):
    def __init__(self, encoder, temperature=0.1):
        super(SimCLR, self).__init__()
        self.encoder = encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(SimCLR, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # View 1 = Original Data
        x1 = data

        # View 2 = Augmented (Masked) Data
        x2 = augment_payload(x1)

        with tf.GradientTape() as tape:
            # Get projections (z) for loss calculation
            _, z1 = self.encoder(x1, training=True)
            _, z2 = self.encoder(x2, training=True)

            # Calculate NT-Xent loss between Original and Augmented
            loss = self.loss_fn(z1, z2, self.temperature)

        gradients = tape.gradient(loss, self.encoder.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.encoder.trainable_variables))
        return {"loss": loss}

def nt_xent_loss(z1, z2, temperature):
    batch_size = tf.shape(z1)[0]
    z = tf.concat([z1, z2], axis=0)
    sim_matrix = tf.matmul(z, z, transpose_b=True) / temperature
    mask = tf.eye(2 * batch_size, dtype=tf.bool)
    sim_matrix = tf.where(mask, -1e9, sim_matrix)
    labels = tf.concat([tf.range(batch_size) + batch_size, tf.range(batch_size)], axis=0)
    return tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(labels, sim_matrix, from_logits=True))

# --- 5. Main Execution ---
def main():
    X = load_data()
    if X is None: return

    # Create Dataset
    dataset = tf.data.Dataset.from_tensor_slices(X)
    dataset = dataset.shuffle(1024).batch(BATCH_SIZE, drop_remainder=True)

    # Initialize
    cnn = get_cnn_encoder(X.shape[1:])
    simclr = SimCLR(cnn, temperature=TEMPERATURE)
    simclr.compile(optimizer=optimizers.Adam(LEARNING_RATE), loss_fn=nt_xent_loss)

    print("\n--- Starting SimCLR Pre-Training (Byte Masking) ---")
    simclr.fit(dataset, epochs=EPOCHS, verbose=1)

    print(f"Saving SimCLR weights to {WEIGHTS_OUTPUT_PATH}")
    # Save only the encoder part for fine-tuning
    simclr.encoder.save_weights(WEIGHTS_OUTPUT_PATH)
    print("Done. Weights generated.")

if __name__ == "__main__":
    main()

Loaded Raw Data: (12736, 7840, 1)

--- Starting SimCLR Pre-Training (Byte Masking) ---
Epoch 1/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 36s 219ms/step - loss: 2.0737
Epoch 2/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 217ms/step - loss: 1.8768
Epoch 3/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 217ms/step - loss: 1.8236
Epoch 4/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 218ms/step - loss: 1.7914
Epoch 5/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 219ms/step - loss: 1.7722
Epoch 6/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 220ms/step - loss: 1.7484
Epoch 7/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 220ms/step - loss: 1.7805
Epoch 8/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 221ms/step - loss: 1.7420
Epoch 9/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 222ms/step - loss: 1.7300
Epoch 10/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 221ms/step - loss: 1.6919
Epoch 11/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 221ms/step - loss: 1.6882
Epoch 12/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 221ms/step - loss: 1.6755
Epoch 13/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 222ms/step - loss: 1.6589
Epoch 14/50
99/99 ━━━━━━━

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.20
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.9306

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.8223

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.8817


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.05
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.8842

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.6521

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.7044


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.10
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.9187

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.7807

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.8550


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.30
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.9351

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.8578

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.8952


FIX

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model, optimizers
import numpy as np
import os

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
WEIGHTS_OUTPUT_PATH = os.path.join(BASE_PATH, "v2SIMCLR_Encoder.weights.h5")

# Config
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.0001
TEMPERATURE = 0.1
PROJECTION_DIM = 64
LATENT_DIM = 128

# --- 1. Data Augmentation (The "Uni-Modal" Trick) ---
def augment_payload(x):
    """
    True Byte Masking: Randomly zeros out 15% of bytes WITHOUT scaling the others.
    """
    # 1. Generate binary mask (1 = keep, 0 = drop)
    # Shape must match x: (Batch, 7840, 1)
    mask = tf.random.categorical(
        tf.math.log([[0.15, 0.85]]), # 15% drop, 85% keep
        num_samples=tf.size(x) // x.shape[-1]
    )
    mask = tf.cast(tf.reshape(mask, tf.shape(x)), dtype=x.dtype)

    # 2. Apply mask (No scaling!)
    return x * mask

# --- 2. Data Loading ---
def load_data():
    if not os.path.exists(RAW_PAYLOAD_PATH):
        print(f"Error: {RAW_PAYLOAD_PATH} not found.")
        return None
    X = np.load(RAW_PAYLOAD_PATH).astype('float32')

    # Reshape for CNN (N, 7840, 1) - Flattening packets to 1 stream
    # Note: Ensure this matches the shape used in your main LE-MVCL
    if len(X.shape) == 3: # If (N, 10, 784)
        X = X.reshape(X.shape[0], -1, 1)
    else:
        X = X.reshape(X.shape[0], -1, 1)

    print(f"Loaded Raw Data: {X.shape}")
    return X

# --- 3. CNN Architecture (Must match LE-MVCL exactly) ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)

    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)

    # Projection Head (Only for training)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1, epsilon=1e-10))(z)

    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 4. SimCLR Model Wrapper ---
class SimCLR(Model):
    def __init__(self, encoder, temperature=0.1):
        super(SimCLR, self).__init__()
        self.encoder = encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(SimCLR, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # Standard SimCLR: Two different random augmentations of the SAME image/flow
        x1 = augment_payload(data) # View A (Random Mask 1)
        x2 = augment_payload(data) # View B (Random Mask 2)

        with tf.GradientTape() as tape:
            _, z1 = self.encoder(x1, training=True)
            _, z2 = self.encoder(x2, training=True)

            loss = self.loss_fn(z1, z2, self.temperature)

        gradients = tape.gradient(loss, self.encoder.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.encoder.trainable_variables))
        return {"loss": loss}

def nt_xent_loss(z1, z2, temperature):
    batch_size = tf.shape(z1)[0]
    z = tf.concat([z1, z2], axis=0)
    sim_matrix = tf.matmul(z, z, transpose_b=True) / temperature
    mask = tf.eye(2 * batch_size, dtype=tf.bool)
    sim_matrix = tf.where(mask, -1e9, sim_matrix)
    labels = tf.concat([tf.range(batch_size) + batch_size, tf.range(batch_size)], axis=0)
    return tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(labels, sim_matrix, from_logits=True))

# --- 5. Main Execution ---
def main():
    X = load_data()
    if X is None: return

    # Create Dataset
    dataset = tf.data.Dataset.from_tensor_slices(X)
    dataset = dataset.shuffle(1024).batch(BATCH_SIZE, drop_remainder=True)

    # Initialize
    cnn = get_cnn_encoder(X.shape[1:])
    simclr = SimCLR(cnn, temperature=TEMPERATURE)
    simclr.compile(optimizer=optimizers.Adam(LEARNING_RATE), loss_fn=nt_xent_loss)

    print("\n--- Starting SimCLR Pre-Training (Byte Masking) ---")
    simclr.fit(dataset, epochs=EPOCHS, verbose=1)

    print(f"Saving SimCLR weights to {WEIGHTS_OUTPUT_PATH}")
    # Save only the encoder part for fine-tuning
    simclr.encoder.save_weights(WEIGHTS_OUTPUT_PATH)
    print("Done. Weights generated.")

if __name__ == "__main__":
    main()

Loaded Raw Data: (12736, 7840, 1)

--- Starting SimCLR Pre-Training (Byte Masking) ---
Epoch 1/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 34s 222ms/step - loss: 2.1111
Epoch 2/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 224ms/step - loss: 1.9464
Epoch 3/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 225ms/step - loss: 1.9482
Epoch 4/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 225ms/step - loss: 1.8652
Epoch 5/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 41s 223ms/step - loss: 1.8793
Epoch 6/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 225ms/step - loss: 1.8261
Epoch 7/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 225ms/step - loss: 1.8333
Epoch 8/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 224ms/step - loss: 1.7806
Epoch 9/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 225ms/step - loss: 1.8083
Epoch 10/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 224ms/step - loss: 1.8138
Epoch 11/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 225ms/step - loss: 1.7671
Epoch 12/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 224ms/step - loss: 1.7697
Epoch 13/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 22s 224ms/step - loss: 1.8022
Epoch 14/50
99/99 ━━━━━━━

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "v2SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.20
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/v2SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.9321

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.8348

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.8783


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "v2SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.10
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/v2SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.9201

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.7641

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.8169


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "v2SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.05
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/v2SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.8819

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.6739

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.7271


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
# We need the LABELS file to map the .npy indices to filenames
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

WEIGHTS_PATH = os.path.join(BASE_PATH, "v2SIMCLR_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.30
LATENT_DIM = 128
PROJECTION_DIM = 64

# --- 1. Define Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    return Model(inputs, [h, z], name="CNN_SimCLR")

# --- 2. Load & Align Data (THE FIX) ---
def load_and_align_data():
    print("--- Loading and Aligning Data ---")

    # 1. Load Raw Payloads (12736 rows)
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None, None
    X_raw_all = np.load(RAW_PAYLOAD_PATH).astype('float32')
    X_raw_all = X_raw_all.reshape(X_raw_all.shape[0], -1, 1) # Reshape to (N, 7840, 1)

    # 2. Load Raw Labels (to get filenames for the .npy rows)
    df_raw_labels = pd.read_csv(RAW_LABELS_PATH)
    # Create a map: filename -> index in X_raw_all
    filename_to_idx = {name: i for i, name in enumerate(df_raw_labels['filename'])}

    # 3. Load Merged Stats CSV (12555 rows - The Target)
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. ALIGNMENT LOOP
    valid_payloads = []
    valid_stats = []

    # Get Stats Columns
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app']
    stats_cols = [c for c in df_merged.columns if (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_')) and not any(x in c for x in exclude)]

    print(f"Aligning... Target size: {len(df_merged)}")

    # We iterate through the CLEAN CSV (df_merged) and grab the corresponding payload
    aligned_indices = []
    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_to_idx:
            # Found the matching payload!
            npy_idx = filename_to_idx[fname]
            valid_payloads.append(X_raw_all[npy_idx])
            valid_stats.append(row[stats_cols].values)
            aligned_indices.append(idx)
        else:
            # This shouldn't happen often, but good to handle
            print(f"Warning: {fname} found in CSV but not in Payload .npy")

    # Convert to Numpy
    X_payload_final = np.array(valid_payloads)
    X_stats_final = np.array(valid_stats).astype('float32')

    # Normalize Stats
    X_stats_final = StandardScaler().fit_transform(X_stats_final)

    # Filter df_merged to match exactly (in case some were missing)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Alignment Complete. Shapes -> Payload: {X_payload_final.shape}, Stats: {X_stats_final.shape}")
    return df_final, X_payload_final, X_stats_final

# --- 3. Trainer Helper ---
def train_task(X, y, task_name):
    print(f"\n>>> SimCLR Baseline: {task_name}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=LABEL_PERCENTAGE, random_state=42, stratify=y)

    weights = compute_sample_weight('balanced', y_train)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
    clf.fit(X_train, y_train, sample_weight=weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    F1 Score: {f1:.4f}")

# --- 4. Main ---
# --- 4. Main (Fixed) ---
def main():
    # 1. Load Data (Aligned)
    df, X_pay, X_stat = load_and_align_data()
    if df is None: return

    # 2. Reconstruct Labels (Crucial Step missing before)
    print("Reconstructing Labels...")
    # Find columns that look like 'application' and 'category'
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    cat_col = next((c for c in df.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print(f"Critical Error: Could not find 'application' or 'category' columns in {df.columns}")
        return

    final_binary, final_category, final_app = [], [], []

    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        # Logic: If filename has "vpn", it's VPN.
        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_binary.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    # Add these new labels to the dataframe
    df['Binary_Label'] = final_binary
    df['Category_Label'] = final_category
    df['App_Label'] = final_app

    # 3. Extract Features using SimCLR Weights
    print(f"Loading SimCLR Weights from: {WEIGHTS_PATH}")
    cnn = get_cnn_encoder(X_pay.shape[1:])

    try:
        cnn.load_weights(WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    # Create extractor (output 'h', index 0)
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    h_simclr = extractor.predict(X_pay, batch_size=128, verbose=0)

    # 4. Fuse with Raw Stats
    # NOW shapes should match
    try:
        X_final = np.concatenate([h_simclr, X_stat], axis=1)
        print(f"Fused Feature Shape: {X_final.shape}")
    except ValueError as e:
        print(f"FATAL ERROR: Shapes still mismatch. {e}")
        return

    # 5. Run Experiments
    le = LabelEncoder()

    # A. Binary Task
    y_bin = le.fit_transform(df['Binary_Label'])
    train_task(X_final, y_bin, "Binary Task")

    # B. Category Task (VPN Only)
    mask_vpn = df['Binary_Label'] == 'VPN'
    y_cat = le.fit_transform(df.loc[mask_vpn, 'Category_Label'])
    train_task(X_final[mask_vpn], y_cat, "VPN Category")

    # C. Apps (Top 6)
    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask_app = df['App_Label'].isin(target_apps)
    y_app = le.fit_transform(df.loc[mask_app, 'App_Label'])
    train_task(X_final[mask_app], y_app, "VPN Top Apps")

if __name__ == "__main__":
    main()

--- Loading and Aligning Data ---
Aligning... Target size: 12555
Alignment Complete. Shapes -> Payload: (12555, 7840, 1), Stats: (12555, 137)
Reconstructing Labels...
Loading SimCLR Weights from: /content/drive/MyDrive/1 Skripsi/27jan/v2SIMCLR_Encoder.weights.h5
Weights loaded successfully.
Fused Feature Shape: (12555, 265)

>>> SimCLR Baseline: Binary Task
    F1 Score: 0.9376

>>> SimCLR Baseline: VPN Category
    F1 Score: 0.8672

>>> SimCLR Baseline: VPN Top Apps
    F1 Score: 0.8831
